# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Предобработаны данные в SQL, и теперь они готовы для проверки гипотезы в Python.

Как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

- Автор:Голубова Елизавета

## Цели и задачи проекта

**Цель:** Проверить гипотезу о том, что пользователи из Санкт-Петербурга проводят больше времени за чтением и прослушиванием книг в приложении Яндекс Книги по сравнению с пользователями из Москвы, используя статистические методы.

**Задачи:**
1. Загрузить предобработанные данные пользователей из Москвы и Санкт-Петербурга
2. Проверить данные на дубликаты и целостность
3. Сравнить размеры групп и их статистические характеристики
4. Провести статистическую проверку гипотезы с помощью t-теста
5. Интерпретировать результаты и подготовить аналитическую записку

## Описание данных

Для анализа используются данные о пользовательской активности в сервисе Яндекс Книги. Данные содержат следующую информацию:

**Структура данных:**
- `city` — город пользователя (Москва или Санкт-Петербург)
- `puid` — уникальный идентификатор пользователя
- `hours` — суммарное количество часов активности пользователя в приложении

**Источник данных:** Таблицы сервиса Яндекс Книги:
- `bookmate.audition` — данные об активности пользователей
- `bookmate.geo` — данные о географическом местоположении

**Период данных:** 1 сентября - 11 декабря 2024 года


## Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка гипотезы в Python
3. Аналитическая записка

---

## 1. Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
# Загружаем библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Загружаем данные
df = pd.read_csv('/datasets/yandex_knigi_data.csv')

In [3]:
# Загружаем первые строки
df.head()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


In [5]:
# Удаляем столбец Unnamed: 0, так как он дублирует индексы
df = df.drop(columns=['Unnamed: 0'])

# Проверяем результат
df.head()

,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


In [6]:
# Проверяем дубликаты по puid
print(f"Дубликатов по puid: {df['puid'].duplicated().sum()}")

# Удаляем дубликаты по puid
df = df.drop_duplicates(subset=['puid'])

print(f"Уникальных пользователей: {len(df)}")

Дубликатов по puid: 244
Уникальных пользователей: 8540


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [7]:
# Разделяем данные по городам
df_moscow = df[df['city'] == 'Москва']
df_spb = df[df['city'] == 'Санкт-Петербург']

print(f"Количество пользователей в Москве: {len(df_moscow)}")
print(f"Количество пользователей в Санкт-Петербурге: {len(df_spb)}")

# Проводим односторонний t-тест (учитывая, что дисперсии могут быть неравны)
alpha = 0.05
t_stat, p_value = stats.ttest_ind(df_spb['hours'], df_moscow['hours'], 
                                  equal_var=False, alternative='greater')

print("\nРезультаты t-теста:")
print(f"t-статистика: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

print("\nСтатистики по группам:")
print(f"Средняя активность в Москве: {df_moscow['hours'].mean():.2f} часов")
print(f"Средняя активность в СПб: {df_spb['hours'].mean():.2f} часов")
print(f"Разница: {df_spb['hours'].mean() - df_moscow['hours'].mean():.2f} часов")

print("\nИнтерпретация результатов:")
if p_value < alpha:
    print(f"p-value ({p_value:.4f}) < α ({alpha}) → Отвергаем H₀")
    print("Есть статистически значимые доказательства, что пользователи из")
    print("Санкт-Петербурга проводят больше времени в приложении.")
else:
    print(f"p-value ({p_value:.4f}) ≥ α ({alpha}) → Не отвергаем H₀")
    print("Нет статистически значимых доказательств различий в активности.")

Количество пользователей в Москве: 6234
Количество пользователей в Санкт-Петербурге: 2306

Результаты t-теста:
t-статистика: 0.4028
p-value: 0.3436

Статистики по группам:
Средняя активность в Москве: 10.88 часов
Средняя активность в СПб: 11.26 часов
Разница: 0.38 часов

Интерпретация результатов:
p-value (0.3436) ≥ α (0.05) → Не отвергаем H₀
Нет статистически значимых доказательств различий в активности.


## 3. Аналитическая записка
По результатам анализа данных подготовим аналитическую записку, в которой опишем:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



1. Выбранный тип t-теста и уровень статистической значимости  
- **Тип теста:** Односторонний t-тест Уэлча 
- **Уровень значимости:** **α = 0.05**  
- **Направление теста:** Проверка гипотезы о том, что средняя активность пользователей в Санкт-Петербурге **больше**, чем в Москве  
- **Причина выбора теста Уэлча:** Тест не требует предположения о равенстве дисперсий в группах

**Обоснование выбора теста Уэлча:** 
При сравнении двух независимых выборок рекомендуется сначала проверить равенство дисперсий. 
Тест Уэлча является более консервативным и надежным вариантом, так как он не предполагает 
равенства дисперсий в группах, что делает его применимым в большинстве практических ситуаций. 
В отличие от классического t-теста Стьюдента, тест Уэлча дает корректные результаты даже при 
неравных дисперсиях и разном размере выборок.


2. Результат теста (p-value)  
- **t-статистика:** **0.7782**  
- **p-value:** **0.2182**  
- **Средняя активность в Москве:** **10.88 часов**  
- **Средняя активность в Санкт-Петербурге:** **11.59 часов**  
- **Разница средних:** **0.71 часов**  

---

3. Вывод на основе полученного p-value  
Так как **p-value (0.2182) ≥ α (0.05)**, мы **не отвергаем нулевую гипотезу H₀**.

**Статистический вывод:** Нет достаточных статистически значимых доказательств того, что пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении Яндекс Книги, чем пользователи из Москвы.

> **Примечание:** Хотя средняя активность в Санкт-Петербурге действительно выше на 0.71 часа, эта разница **не является статистически значимой** на уровне значимости 5%.

---

4. Возможные причины, объясняющие полученные результаты  

1. **Недостаточная мощность теста**  
   Несмотря на общий объем данных (8,784 пользователя), распределение по городам неравномерно:
   - Москва: **6,234 пользователя**  
   - Санкт-Петербург: **2,550 пользователей**  
   
   Меньший размер выборки Санкт-Петербурга может ограничивать возможность обнаружения статистически значимых различий.

2. **Высокая вариативность данных**  
   Время активности пользователей может сильно варьироваться в зависимости от множества факторов:
   - Сезонность  
   - День недели  
   - Личные предпочтения  
   
   Это создает "шум" в данных и затрудняет выявление устойчивых различий между городами.

3. **Схожесть пользовательского поведения**  
   Возможно, что культурные и демографические различия между жителями Москвы и Санкт-Петербурга в контексте использования цифровых сервисов чтения **менее выражены**, чем предполагалось изначально.


----

# Часть 2. Анализ результатов A/B-тестирования

Интернет-магазин BitMotion Kit продаёт геймифицированные товары для здорового образа жизни (эспандер со счётчиком, подстольный велотренажёр с Bluetooth). Отзывы пользователей указывают на то, что интерфейс магазина слишком сложен.

Для решения этой проблемы была разработана новая версия сайта и проведён A/B-тест: часть пользователей видела новый интерфейс, часть — старый. Цель — проверить, повышает ли новый интерфейс конверсию в покупку.

В распоряжении: данные о действиях пользователей, распределение по группам, техническое задание.

## 1. Описание цели исследования.



**Цель:**  
Оценить эффективность нового интерфейса онлайн-магазина BitMotion Kit и принять обоснованное решение о его внедрении на основе результатов A/B-тестирования.

Задачи:

1. **Проверить корректность проведения A/B-теста:**
   - **Убедиться в правильности разделения пользователей** на группы A (контрольная) и B (тестовая)
   - **Проверить равномерность распределения** по ключевым характеристикам:
     - Устройства (desktop, mobile)
     - Географические регионы (EU, CIS, APAC, N.America)
   - **Оценить соответствие теста техническому заданию:**
     - Название теста: `interface_eu_test`
     - Целевая метрика: конверсия зарегистрированных пользователей в покупателей
     - Временное окно: 7 дней после регистрации
     - Минимальный детектируемый эффект (MDE): **+3 процентных пункта**

2. **Оценить влияние нового интерфейса на ключевую бизнес-метрику:**
   - **Целевая метрика:** Конверсия зарегистрированных пользователей в покупателей в течение 7 дней после регистрации
   - **Гипотеза:** Новый интерфейс (группа B) увеличит конверсию **как минимум на 3 процентных пункта** по сравнению со старым интерфейсом (группа A)

3. **Проанализировать статистическую значимость результатов:**
   - **Определить статистическую значимость** наблюдаемых различий
   - **Рассчитать p-value** для принятия обоснованного решения
   - **Использовать уровень значимости α = 0.05** для проверки гипотезы
   - **Применить z-тест пропорций** для сравнения долей между группами

4. **Сформулировать рекомендации для бизнеса:**
   - **Рекомендовать внедрение** нового интерфейса, если результаты статистически значимы и превышают MDE
   - **Рекомендовать отказ** от изменений, если эффект не подтвержден
   - **Предложить дальнейшие шаги** для оптимизации пользовательского опыта

## 2. Загрузим данные, оценим их целостность.


In [8]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [9]:
# Проверяем дубликаты в событиях
print(f"Всего событий: {len(events)}")
print(f"Дубликатов событий: {events.duplicated().sum()}")

# Удаляем дубликаты событий
events = events.drop_duplicates()
print(f"Событий после удаления дубликатов: {len(events)}")

Всего событий: 787286
Дубликатов событий: 36318
Событий после удаления дубликатов: 750968


## 3. По таблице `ab_test_participants` оценим корректность проведения теста:

   3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [10]:
# Выделяем пользователей, участвующих в тесте interface_eu_test
test_participants = participants[participants['ab_test'] == 'interface_eu_test']

In [11]:
# 1. Соответствие требованиям ТЗ
print("1. СООТВЕТСТВИЕ ТЕХНИЧЕСКОМУ ЗАДАНИЮ:")
print(f"Название теста: 'interface_eu_test' - {'СООТВЕТСТВУЕТ' if 'interface_eu_test' in participants['ab_test'].unique() else 'НЕ СООТВЕТСТВУЕТ'}")
print(f"Группы: A и B - {'СООТВЕТСТВУЕТ' if set(['A', 'B']).issubset(set(test_participants['group'].unique())) else 'НЕ СООТВЕТСТВУЕТ'}")
print(f"Всего пользователей в тесте: {len(test_participants)}")

1. СООТВЕТСТВИЕ ТЕХНИЧЕСКОМУ ЗАДАНИЮ:
Название теста: 'interface_eu_test' - СООТВЕТСТВУЕТ
Группы: A и B - СООТВЕТСТВУЕТ
Всего пользователей в тесте: 10850


In [12]:
# 2. Равномерность распределения по группам
print("2. РАВНОМЕРНОСТЬ РАСПРЕДЕЛЕНИЯ ПО ГРУППАМ:")
group_counts = test_participants['group'].value_counts().sort_index()
print("Количество пользователей по группам:")
for group in ['A', 'B']:
    count = group_counts.get(group, 0)
    percentage = (count / len(test_participants) * 100) if len(test_participants) > 0 else 0
    print(f"Группа {group}: {count} пользователей ({percentage:.1f}%)")

2. РАВНОМЕРНОСТЬ РАСПРЕДЕЛЕНИЯ ПО ГРУППАМ:
Количество пользователей по группам:
Группа A: 5383 пользователей (49.6%)
Группа B: 5467 пользователей (50.4%)


In [13]:
# 3. Отсутствие пересечений с конкурирующим тестом
print("3. ОТСУТСТВИЕ ПЕРЕСЕЧЕНИЙ С КОНКУРИРУЮЩИМ ТЕСТОМ:")

# Проверяем, есть ли пользователи в нескольких тестах
user_test_counts = participants.groupby('user_id')['ab_test'].nunique()
multi_test_users = user_test_counts[user_test_counts > 1]

print(f"Пользователей в нескольких тестах: {len(multi_test_users)}")

3. ОТСУТСТВИЕ ПЕРЕСЕЧЕНИЙ С КОНКУРИРУЮЩИМ ТЕСТОМ:
Пользователей в нескольких тестах: 887


In [14]:
# Проверяем пересечения между группами A и B
print(f"Пользователей в обеих группах A и B: {len(set(test_participants[test_participants['group'] == 'A']['user_id']) & set(test_participants[test_participants['group'] == 'B']['user_id']))}")

Пользователей в обеих группах A и B: 0


In [15]:
# Удаляем пользователей в нескольких тестах
single_test_users = participants.groupby('user_id')['ab_test'].nunique()
single_test_users = single_test_users[single_test_users == 1].index
test_participants = test_participants[test_participants['user_id'].isin(single_test_users)]

print(f"Пользователей после очистки: {len(test_participants)}")

Пользователей после очистки: 9963


3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [16]:
# Получаем список пользователей из теста interface_eu_test
test_user_ids = set(test_participants['user_id'])

# Оставляем только события этих пользователей
test_events = events[events['user_id'].isin(test_user_ids)]

print(f"Всего событий в таблице events: {len(events)}")
print(f"Событий пользователей из теста interface_eu_test: {len(test_events)}")

Всего событий в таблице events: 750968
Событий пользователей из теста interface_eu_test: 68074


- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [17]:
# Находим даты регистрации для каждого пользователя
registration_dates = test_events[test_events['event_name'] == 'registration'].groupby('user_id')['event_dt'].min()

# Добавляем дату регистрации к каждому событию
test_events = test_events.merge(registration_dates.rename('registration_date'), on='user_id')

# Рассчитываем лайфтайм (разницу в днях между событием и регистрацией)
test_events['lifetime_days'] = (test_events['event_dt'] - test_events['registration_date']).dt.days

# Оставляем только события в течение первых 7 дней
test_events = test_events[test_events['lifetime_days'] < 7]

print(f"Событий в течение первых 7 дней после регистрации: {len(test_events)}")

Событий в течение первых 7 дней после регистрации: 58692


Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [18]:
# Задаем параметры
alpha = 0.05  # уровень значимости (1 - 0.95)
power = 0.8   # мощность теста
p = 0.3       # базовый уровень конверсии (30%)
mde = 0.03  # минимальный детектируемый эффект

# Рассчитываем размер эффекта
effect_size = proportion_effectsize(p, p + mde)

# Рассчитываем необходимый размер выборки
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

print(f"Необходимый размер выборки на каждую группу: {int(sample_size)} пользователей")
print(f"Общий необходимый размер выборки: {int(sample_size * 2)} пользователей")
print(f"Текущий размер выборки в тесте: {len(test_participants)} пользователей")
print(f"Группа A: {group_counts.get('A', 0)} пользователей")
print(f"Группа B: {group_counts.get('B', 0)} пользователей")

Необходимый размер выборки на каждую группу: 3761 пользователей
Общий необходимый размер выборки: 7523 пользователей
Текущий размер выборки в тесте: 9963 пользователей
Группа A: 5383 пользователей
Группа B: 5467 пользователей


- рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [19]:
# Получаем список пользователей, совершивших покупку в течение 7 дней
purchasing_users = test_events[test_events['event_name'] == 'purchase']['user_id'].unique()

# Добавляем информацию о группе для каждого пользователя
users_with_group = test_participants[['user_id', 'group']]

# Считаем количество уникальных пользователей по группам
total_users_by_group = users_with_group.groupby('group')['user_id'].nunique()

# Считаем количество покупателей по группам
purchasing_users_by_group = users_with_group[users_with_group['user_id'].isin(purchasing_users)].groupby('group')['user_id'].nunique()

print("Результаты по группам:")
for group in ['A', 'B']:
    total = total_users_by_group.get(group, 0)
    purchasers = purchasing_users_by_group.get(group, 0)
    print(f"Группа {group}:")
    print(f"  Всего пользователей: {total}")
    print(f"  Сделавших покупку: {purchasers}")

Результаты по группам:
Группа A:
  Всего пользователей: 4952
  Сделавших покупку: 1377
Группа B:
  Всего пользователей: 5011
  Сделавших покупку: 1480


- сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

In [20]:
# Рассчитываем конверсию для каждой группы
conversion_a = purchasing_users_by_group.get('A', 0) / total_users_by_group.get('A', 1)
conversion_b = purchasing_users_by_group.get('B', 0) / total_users_by_group.get('B', 1)

# Рассчитываем абсолютную и относительную разницу
abs_difference = conversion_b - conversion_a
rel_difference = (conversion_b - conversion_a) / conversion_a * 100

print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ ПО ГРУППАМ:")
print(f"Конверсия в группе A (контрольная): {conversion_a:.3f} ({conversion_a*100:.1f}%)")
print(f"Конверсия в группе B (тестовая): {conversion_b:.3f} ({conversion_b*100:.1f}%)")
print(f"Абсолютная разница: {abs_difference:.3f} ({abs_difference*100:.1f} п.п.)")
print(f"Относительная разница: {rel_difference:.1f}%")

СРАВНЕНИЕ РЕЗУЛЬТАТОВ ПО ГРУППАМ:
Конверсия в группе A (контрольная): 0.278 (27.8%)
Конверсия в группе B (тестовая): 0.295 (29.5%)
Абсолютная разница: 0.017 (1.7 п.п.)
Относительная разница: 6.2%


## 4. Проведем оценку результатов A/B-тестирования:

- Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [21]:
# Данные для теста
counts = [purchasing_users_by_group.get('A', 0), purchasing_users_by_group.get('B', 0)]
nobs = [total_users_by_group.get('A', 0), total_users_by_group.get('B', 0)]

# Проводим z-тест пропорций (односторонний)
alpha = 0.05
z_stat, p_value = proportions_ztest(
    [purchasing_users_by_group.get('B', 0), purchasing_users_by_group.get('A', 0)],  # B, A
    [total_users_by_group.get('B', 0), total_users_by_group.get('A', 0)],            # B, A
    alternative='larger'
)

print(f"z-статистика: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Уровень значимости α: {alpha}")

if p_value < alpha:
    print(f"p-value ({p_value:.4f}) < α ({alpha})")
else:
    print(f"p-value ({p_value:.4f}) ≥ α ({alpha})")

z-статистика: 1.9070
p-value: 0.0283
Уровень значимости α: 0.05
p-value (0.0283) < α (0.05)


- Опишем выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

**1. Статистические результаты теста:**
- **z-статистика:** 1.9070
- **p-value:** 0.0283
- **Уровень значимости:** α = 0.05
- **Сравнение:** p-value (0.0283) < α (0.05)

**2. Статистический вывод:**
**Отвергаем нулевую гипотезу H₀.** Есть статистически значимые доказательства того, что конверсия в группе B (новый интерфейс) выше, чем в группе A (старый интерфейс) на уровне значимости 5%.

**3. Оценка ожидаемого эффекта:**
**Требуемый эффект по ТЗ:** Увеличение конверсии на **минимум 3 процентных пункта**

**Фактический результат:** 
- Конверсия в группе A: **27.8%**
- Конверсия в группе B: **29.5%**
- Улучшение: **+1.7 процентных пункта** (+6.2% относительно)

**Вывод по эффекту:** Новый интерфейс **не достиг** требуемого минимального эффекта в 3 процентных пункта, хотя показал статистически значимое улучшение.

**4. Общие выводы по результатам A/B-тестирования:**

- **Статистическая значимость:** Разница в конверсии между группами статистически значима на уровне α = 0.05.

- **Недостаточность эффекта:** Несмотря на статистическую значимость, фактическое улучшение (+1.7 п.п.) не достигает целевого показателя в 3 процентных пункта.

- **Достаточность выборки:** Текущий размер выборки (9,963 пользователя) превышает необходимый минимальный (7,523 пользователя), что обеспечивает достаточную статистическую мощность.

- **Корректность теста:** После очистки данных от пользователей, участвующих в нескольких тестах, проведение теста можно считать корректным.

**Итоговый вывод:** Несмотря на **статистически значимое улучшение** конверсии, новый интерфейс **не рекомендуется** к внедрению, так как:
- Фактический эффект (+1.7 п.п.) не достигает целевого показателя в 3 процентных пункта
- Хотя улучшение статистически значимо, оно недостаточно для достижения бизнес-целей